<a href="https://colab.research.google.com/github/Chris02374/Kashin-IS-27/blob/main-kashin/PZ15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Практическая работа №15
Вариант 8

Приложение ЗАКАЗЫ ТОВАРОВ для автоматизированного контроля заказов торговой фирмы. Таблица Заказы должна содержать информацию о товарах со следующей структурой записи: Код товара, Наименование товара, Заказчик (наименование организации, возможны повторяющиеся значения), Дата заказа, Срок исполнения (от 1 до 10 дней), Стоимость заказа.

In [ ]:
import sqlite3
from datetime import datetime, timedelta


def create_db_and_table():
    """Создание базы данных и таблицы 'Заказы'"""
    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Заказы (
            Код_товара INTEGER PRIMARY KEY,
            Наименование_товара TEXT NOT NULL,
            Заказчик TEXT NOT NULL,
            Дата_заказа TEXT NOT NULL,
            Срок_исполнения INTEGER CHECK (Срок_исполнения BETWEEN 1 AND 10),
            Стоимость_заказа REAL CHECK (Стоимость_заказа >= 0)
        )
    ''')
    conn.commit()
    conn.close()
    print("✅ База данных и таблица 'Заказы' созданы/проверены.")


def insert_sample_data():
    """Ввод 10 записей в таблицу"""
    sample_orders = [
        (101, "Ноутбук Lenovo", "ООО ТехноПлюс", "2024-01-15", 5, 45000.00),
        (102, "Монитор Samsung", "ИП Иванов", "2024-01-20", 3, 25000.00),
        (103, "Клавиатура Logitech", "ООО ТехноПлюс", "2024-02-01", 7, 3500.00),
        (104, "Мышь Logitech", "ЗАО Компьютерный мир", "2024-02-10", 10, 1500.00),
        (105, "Принтер HP", "ООО ТехноПлюс", "2024-02-15", 4, 12000.00),
        (106, "Сканер Canon", "ИП Иванов", "2024-03-01", 6, 8000.00),
        (107, "Веб-камера Logitech", "ЗАО Компьютерный мир", "2024-03-10", 8, 4500.00),
        (108, "Наушники Sony", "ООО Электроника", "2024-03-15", 2, 5500.00),
        (109, "Внешний SSD 1TB", "ООО ТехноПлюс", "2024-04-01", 9, 8900.00),
        (110, "Чехол для ноутбука", "ИП Петров", "2024-04-05", 1, 1200.00),
    ]

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    try:
        cursor.executemany('''
            INSERT OR REPLACE INTO Заказы
            (Код_товара, Наименование_товара, Заказчик, Дата_заказа, Срок_исполнения, Стоимость_заказа)
            VALUES (?, ?, ?, ?, ?, ?)
        ''', sample_orders)
        conn.commit()
        print("✅ 10 записей успешно добавлены в таблицу 'Заказы'.")
    except sqlite3.Error as e:
        print(f"❌ Ошибка при добавлении данных: {e}")
    finally:
        conn.close()


def show_all_orders():
    """Вывод всех заказов на экран"""
    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM Заказы ORDER BY Код_товара")
    orders = cursor.fetchall()
    conn.close()

    if not orders:
        print("📭 В таблице нет записей.")
        return

    print("\n" + "=" * 90)
    print(f"{'Код':<8} {'Наименование':<25} {'Заказчик':<22} {'Дата заказа':<12} {'Срок':<6} {'Стоимость':<12}")
    print("-" * 90)
    for order in orders:
        print(f"{order[0]:<8} {order[1]:<25} {order[2]:<22} {order[3]:<12} {order[4]:<6} {order[5]:<12.2f}")
    print("=" * 90)
    print(f"Всего заказов: {len(orders)}")


# ============= ПОИСК (3 SQL-запроса с WHERE) =============

def search_by_customer():
    """Поиск заказов по заказчику (полное совпадение)"""
    customer = input("Введите наименование заказчика для поиска: ").strip()
    if not customer:
        print("❌ Наименование не может быть пустым.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM Заказы WHERE Заказчик = ?", (customer,))
    orders = cursor.fetchall()
    conn.close()

    if orders:
        print(f"\n✅ Найдено заказов у заказчика '{customer}': {len(orders)}")
        for o in orders:
            print(f"   Код:{o[0]}, Товар:{o[1]}, Дата:{o[3]}, Стоимость:{o[5]:.2f}")
    else:
        print(f"❌ Заказы от заказчика '{customer}' не найдены.")


def search_by_price_range():
    """Поиск заказов по диапазону стоимости"""
    try:
        min_price = float(input("Введите минимальную стоимость: "))
        max_price = float(input("Введите максимальную стоимость: "))
        if min_price > max_price:
            print("❌ Минимальная стоимость не может быть больше максимальной.")
            return
    except ValueError:
        print("❌ Введите корректные числа.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM Заказы WHERE Стоимость_заказа BETWEEN ? AND ?", (min_price, max_price))
    orders = cursor.fetchall()
    conn.close()

    if orders:
        print(f"\n✅ Найдено заказов в диапазоне [{min_price}, {max_price}]: {len(orders)}")
        for o in orders:
            print(f"   Код:{o[0]}, Товар:{o[1]}, Заказчик:{o[2]}, Стоимость:{o[5]:.2f}")
    else:
        print(f"❌ Заказы в диапазоне [{min_price}, {max_price}] не найдены.")


def search_by_deadline():
    """Поиск заказов по сроку исполнения (больше или равно заданному)"""
    try:
        days = int(input("Введите минимальный срок исполнения (дней): "))
        if days < 1 or days > 10:
            print("❌ Срок исполнения должен быть от 1 до 10 дней.")
            return
    except ValueError:
        print("❌ Введите целое число.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM Заказы WHERE Срок_исполнения >= ?", (days,))
    orders = cursor.fetchall()
    conn.close()

    if orders:
        print(f"\n✅ Найдено заказов со сроком исполнения >= {days} дней: {len(orders)}")
        for o in orders:
            print(f"   Код:{o[0]}, Товар:{o[1]}, Срок:{o[4]} дн., Стоимость:{o[5]:.2f}")
    else:
        print(f"❌ Заказы со сроком исполнения >= {days} дней не найдены.")


# ============= УДАЛЕНИЕ (3 SQL-запроса с WHERE) =============

def delete_by_code():
    """Удаление заказа по коду товара"""
    try:
        code = int(input("Введите код товара для удаления: "))
    except ValueError:
        print("❌ Введите целое число.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM Заказы WHERE Код_товара = ?", (code,))
    order = cursor.fetchone()
    if not order:
        print(f"❌ Заказ с кодом {code} не найден.")
        conn.close()
        return

    confirm = input(f"Удалить заказ '{order[1]}' (заказчик {order[2]})? (да/нет): ").lower()
    if confirm == 'да':
        cursor.execute("DELETE FROM Заказы WHERE Код_товара = ?", (code,))
        conn.commit()
        print(f"✅ Заказ с кодом {code} удалён.")
    else:
        print("❌ Удаление отменено.")
    conn.close()


def delete_by_customer():
    """Удаление всех заказов указанного заказчика"""
    customer = input("Введите наименование заказчика для удаления всех его заказов: ").strip()
    if not customer:
        print("❌ Наименование не может быть пустым.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM Заказы WHERE Заказчик = ?", (customer,))
    count = cursor.fetchone()[0]
    if count == 0:
        print(f"❌ Заказы от заказчика '{customer}' не найдены.")
        conn.close()
        return

    confirm = input(f"Удалить ВСЕ заказы заказчика '{customer}' ({count} записей)? (да/нет): ").lower()
    if confirm == 'да':
        cursor.execute("DELETE FROM Заказы WHERE Заказчик = ?", (customer,))
        conn.commit()
        print(f"✅ Удалено {count} заказ(ов) от заказчика '{customer}'.")
    else:
        print("❌ Удаление отменено.")
    conn.close()


def delete_by_date_before():
    """Удаление заказов с датой ранее указанной"""
    date_str = input("Введите дату в формате ГГГГ-ММ-ДД (удалить заказы ранее этой даты): ").strip()
    try:
        datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError:
        print("❌ Неверный формат даты. Используйте ГГГГ-ММ-ДД")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM Заказы WHERE Дата_заказа < ?", (date_str,))
    count = cursor.fetchone()[0]
    if count == 0:
        print(f"❌ Заказов с датой ранее {date_str} не найдено.")
        conn.close()
        return

    confirm = input(f"Удалить {count} заказ(ов) с датой ранее {date_str}? (да/нет): ").lower()
    if confirm == 'да':
        cursor.execute("DELETE FROM Заказы WHERE Дата_заказа < ?", (date_str,))
        conn.commit()
        print(f"✅ Удалено {count} заказ(ов).")
    else:
        print("❌ Удаление отменено.")
    conn.close()


# ============= РЕДАКТИРОВАНИЕ (3 SQL-запроса с WHERE) =============

def update_price_by_code():
    """Изменение стоимости заказа по коду товара"""
    try:
        code = int(input("Введите код товара: "))
        new_price = float(input("Введите новую стоимость: "))
        if new_price < 0:
            print("❌ Стоимость не может быть отрицательной.")
            return
    except ValueError:
        print("❌ Введите корректные числа.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT Наименование_товара FROM Заказы WHERE Код_товара = ?", (code,))
    order = cursor.fetchone()
    if not order:
        print(f"❌ Заказ с кодом {code} не найден.")
        conn.close()
        return

    cursor.execute("UPDATE Заказы SET Стоимость_заказа = ? WHERE Код_товара = ?", (new_price, code))
    conn.commit()
    print(f"✅ Стоимость товара '{order[0]}' (код {code}) изменена на {new_price:.2f}")
    conn.close()


def update_deadline_by_customer():
    """Изменение срока исполнения для всех заказов указанного заказчика"""
    customer = input("Введите наименование заказчика: ").strip()
    try:
        new_deadline = int(input("Введите новый срок исполнения (1-10 дней): "))
        if new_deadline < 1 or new_deadline > 10:
            print("❌ Срок исполнения должен быть от 1 до 10 дней.")
            return
    except ValueError:
        print("❌ Введите целое число.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT COUNT(*) FROM Заказы WHERE Заказчик = ?", (customer,))
    count = cursor.fetchone()[0]
    if count == 0:
        print(f"❌ Заказы от заказчика '{customer}' не найдены.")
        conn.close()
        return

    cursor.execute("UPDATE Заказы SET Срок_исполнения = ? WHERE Заказчик = ?", (new_deadline, customer))
    conn.commit()
    print(f"✅ У {count} заказ(ов) заказчика '{customer}' срок исполнения изменён на {new_deadline} дн.")
    conn.close()


def update_product_name_by_code():
    """Изменение наименования товара по коду"""
    try:
        code = int(input("Введите код товара: "))
        new_name = input("Введите новое наименование товара: ").strip()
        if not new_name:
            print("❌ Наименование не может быть пустым.")
            return
    except ValueError:
        print("❌ Введите корректный код.")
        return

    conn = sqlite3.connect('orders.db')
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM Заказы WHERE Код_товара = ?", (code,))
    order = cursor.fetchone()
    if not order:
        print(f"❌ Заказ с кодом {code} не найден.")
        conn.close()
        return

    cursor.execute("UPDATE Заказы SET Наименование_товара = ? WHERE Код_товара = ?", (new_name, code))
    conn.commit()
    print(f"✅ Наименование товара для кода {code} изменено с '{order[1]}' на '{new_name}'")
    conn.close()


# ============= ГЛАВНОЕ МЕНЮ =============

def main_menu():
    """Главное меню программы"""
    while True:
        print("\n" + "=" * 60)
        print("         ПРОГРАММА УПРАВЛЕНИЯ ЗАКАЗАМИ")
        print("=" * 60)
        print("1.  Показать все заказы")
        print("\n--- ПОИСК (3 запроса с WHERE) ---")
        print("2.  Поиск по заказчику")
        print("3.  Поиск по диапазону стоимости")
        print("4.  Поиск по сроку исполнения (>= дней)")
        print("\n--- УДАЛЕНИЕ (3 запроса с WHERE) ---")
        print("5.  Удаление по коду товара")
        print("6.  Удаление всех заказов заказчика")
        print("7.  Удаление заказов ранее указанной даты")
        print("\n--- РЕДАКТИРОВАНИЕ (3 запроса с WHERE) ---")
        print("8.  Изменение стоимости по коду товара")
        print("9.  Изменение срока исполнения у заказчика")
        print("10. Изменение наименования товара по коду")
        print("\n0.  Выход")
        print("-" * 60)

        choice = input("Выберите пункт меню: ").strip()

        if choice == '1':
            show_all_orders()
        elif choice == '2':
            search_by_customer()
        elif choice == '3':
            search_by_price_range()
        elif choice == '4':
            search_by_deadline()
        elif choice == '5':
            delete_by_code()
        elif choice == '6':
            delete_by_customer()
        elif choice == '7':
            delete_by_date_before()
        elif choice == '8':
            update_price_by_code()
        elif choice == '9':
            update_deadline_by_customer()
        elif choice == '10':
            update_product_name_by_code()
        elif choice == '0':
            print("👋 Программа завершена. До свидания!")
            break
        else:
            print("❌ Неверный выбор. Попробуйте снова.")


# Запуск программы
if __name__ == "__main__":
    create_db_and_table()
    insert_sample_data()
    print("\n📋 Исходные данные в таблице:")
    show_all_orders()
    main_menu()

✅ База данных и таблица 'Заказы' созданы/проверены.
✅ 10 записей успешно добавлены в таблицу 'Заказы'.

📋 Исходные данные в таблице:

Код      Наименование              Заказчик               Дата заказа  Срок   Стоимость   
------------------------------------------------------------------------------------------
101      Ноутбук Lenovo            ООО ТехноПлюс          2024-01-15   5      45000.00    
102      Монитор Samsung           ИП Иванов              2024-01-20   3      25000.00    
103      Клавиатура Logitech       ООО ТехноПлюс          2024-02-01   7      3500.00     
104      Мышь Logitech             ЗАО Компьютерный мир   2024-02-10   10     1500.00     
105      Принтер HP                ООО ТехноПлюс          2024-02-15   4      12000.00    
106      Сканер Canon              ИП Иванов              2024-03-01   6      8000.00     
107      Веб-камера Logitech       ЗАО Компьютерный мир   2024-03-10   8      4500.00     
108      Наушники Sony             ООО Электрон